In [ ]:
import open3d as o3d
import numpy as np

# 创建点和标签
points = np.random.rand(100, 3)
labels = np.zeros(100, dtype=np.int32)
labels[:10] = 1  # 前10个点打上标签1

# 按标签分配颜色
colors = np.zeros((len(labels), 3))
colors[labels == 0] = [0.6, 0.6, 0.6]
colors[labels == 1] = [1.0, 0, 0]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([pcd])

o3d.visualization.draw_plotly([pcd], width=600, height=400)

In [ ]:
# 改进的滚球方法重建网格，效果最好

import open3d as o3d
import numpy as np
import trimesh
from scipy.spatial import cKDTree


def point_cloud_to_mesh_2(pcd, verbose=False):

    if len(pcd.points) > 200000:
        pcd = pcd.voxel_down_sample(voxel_size=0.05)

    # 法线估计
    #pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.001, max_nn=30))

    # 球枢轴算法的半径范围
    #radii = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.25]

    # 法线估计半径: 用 2%~5% 的最大跨度
    normal_radius = max_span * 0.0005
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=normal_radius, max_nn=2))
    if verbose:
        print(f"[info] 法线估计半径:{normal_radius:.3f}")

    # ball pivoting 半径：0.01~0.5倍最大跨度多组
    radii = [max_span * f for f in [0.0005, 0.003, 0.006, 0.01, 0.015, 0.02]]
    if verbose:
        print(f"[info] 自动ball pivoting半径: {radii}")
    
    # 使用球枢轴算法生成网格
    mesh1 = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd, o3d.utility.DoubleVector(radii)
    )
    tmesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh1)
    tmesh_filled = tmesh.fill_holes(hole_size=0.2)
    mesh1 = tmesh_filled.to_legacy()
    
    vertices = np.asarray(mesh1.vertices)
    faces = np.asarray(mesh1.triangles)
    tm = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    # 修补单三角和单 quad 孔洞
    trimesh.repair.fill_holes(tm)
    tm.remove_unreferenced_vertices()
    
    # Step 3：转换回 Open3D TriangleMesh
    mesh2 = o3d.geometry.TriangleMesh(
        vertices=o3d.utility.Vector3dVector(tm.vertices),
        triangles=o3d.utility.Vector3iVector(tm.faces)
    )

    return mesh2
    
# 加载点云并预处理
#pcd = o3d.io.read_point_cloud('/data/Real3D-AD-PCD/chicken/test/620_good.pcd')
# pcd = o3d.io.read_point_cloud('/data/Real3D-AD-PCD/toffees/test/564_sink.pcd')
#pcd = o3d.io.read_point_cloud('/data/Real3D-AD-PCD/starfish/test/506_bulges_cut.pcd')
# pcd = o3d.io.read_point_cloud('/data/Real3D-AD-PCD/airplane/test/461_bulge_cut.pcd')
pcd = o3d.io.read_point_cloud('/data/Real3D-AD-PCD/airplane/test/536_bulge.pcd')
# pcd = pcd.voxel_down_sample(voxel_size=0.05)

pts = np.asarray(pcd.points)
span = pts.max(axis=0) - pts.min(axis=0)
max_span = span.max()
min_span = span.min()
pt_count = len(pts)
volume = np.prod(span)
density = pt_count / volume if volume > 0 else 0

print(f"[info] 点数:{pt_count}, 跨度:{span}, 最大跨度:{max_span:.3f}")
print(f"[info] 点云包围盒体积: {volume:.3f}")
print(f"[info] 点密度: {density:.3f} 点/单位体积")

mesh2 = point_cloud_to_mesh_2(pcd, True)

# 可视化
o3d.visualization.draw_plotly([mesh2], width=1000, height=600)

In [ ]:
from collections import defaultdict

def mesh_to_point_cloud(mesh, n_global=1024, n_feature=1024):
    """
    将 mesh 网格，进行均匀采样、曲率计算、边界检测等操作，最终返回点云。
    参数:
    - mesh: open3d.geometry.TriangleMesh
    - n_global: 全局均匀采样的点数
    - n_feature: 特征面片的二次采样点数
    返回:
    - pcd: open3d.geometry.PointCloud 点云对象
    """
    assert isinstance(mesh, o3d.geometry.TriangleMesh), "mesh 类型必须为 open3d.geometry.TriangleMesh"

    mesh.compute_vertex_normals()

    # ---- 全局均匀采样 ----
    points = np.asarray(mesh.sample_points_uniformly(n_global).points)  # (n_global, 3)

    # ---- 面法线批量计算（与原版等价，只是批量化） ----
    triangles = np.asarray(mesh.triangles)
    vertices = np.asarray(mesh.vertices)
    v0, v1, v2 = vertices[triangles[:,0]], vertices[triangles[:,1]], vertices[triangles[:,2]]
    face_normals = np.cross(v1 - v0, v2 - v0)
    face_normals /= (np.linalg.norm(face_normals, axis=1, keepdims=True) + 1e-12)
    # print('1')

    # ---- 构建面邻接表（原版逻辑不变） ----
    face_adj = defaultdict(set)
    edge_to_faces = defaultdict(list)
    for i, tri in enumerate(triangles):
        edges = [(tri[0], tri[1]), (tri[1], tri[2]), (tri[2], tri[0])]
        for e in edges:
            key = tuple(sorted(e))
            edge_to_faces[key].append(i)
    for faces in edge_to_faces.values():
        if len(faces) == 2:
            f1, f2 = faces
            face_adj[f1].add(f2)
            face_adj[f2].add(f1)
    # print('2')

    # ---- 曲率估计（面法线夹角均方根，保持原有 for 逻辑） ----
    n_faces = len(triangles)
    curvature = np.zeros(n_faces)
    for i in range(n_faces):
        neighbors = list(face_adj[i])
        if not neighbors:
            curvature[i] = 0
            continue
        angles = []
        for j in neighbors:
            cos_angle = np.dot(face_normals[i], face_normals[j])
            cos_angle = np.clip(cos_angle, -1, 1)
            angle = np.arccos(cos_angle)
            angles.append(angle ** 2)
        curvature[i] = np.sqrt(np.mean(angles)) if angles else 0

    curvature_thr = np.percentile(curvature, 90)
    high_curv_faces = np.where(curvature > curvature_thr)[0]
    # print('3')

    # ---- 边界面片检测 ----
    boundary_edges = [e for e, fs in edge_to_faces.items() if len(fs) == 1]
    boundary_face_indices = set()
    for e in boundary_edges:
        f = edge_to_faces[e][0]
        boundary_face_indices.add(f)
    boundary_faces = np.array(list(boundary_face_indices), dtype=int)

    # ---- 合并特征面片 ----
    feature_faces = np.unique(np.concatenate([high_curv_faces, boundary_faces]))
    feature_triangles = triangles[feature_faces]
    feature_points = []

    # 特征区域高密度采样
    for _ in range(n_feature):
        i = np.random.randint(0, len(feature_triangles))
        tri_idx = feature_faces[i]
        tri = vertices[triangles[tri_idx]]
        r1, r2 = np.random.rand(2)
        if r1 + r2 > 1:
            r1, r2 = 1 - r1, 1 - r2
        a, b, c = tri
        pt = (1 - r1 - r2) * a + r1 * b + r2 * c
        feature_points.append(pt)
    feature_points = np.array(feature_points, dtype=np.float32)

    all_points = np.concatenate([points, feature_points], axis=0)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(all_points)

    return pcd


pcd_2 = mesh_to_point_cloud(mesh2, n_global=8192, n_feature=8192)
o3d.visualization.draw_plotly([mesh2, pcd_2], width=1000, height=600)

In [ ]:
import numpy as np
import open3d as o3d
from collections import defaultdict, deque
from typing import List, Dict, Iterable, Optional, Tuple

# ------------------------ 邻接 / 几何基础 ------------------------

def _face_adjacency(tris: np.ndarray):
    e2f = defaultdict(list)
    for i, t in enumerate(tris):
        for e in ((t[0], t[1]), (t[1], t[2]), (t[2], t[0])):
            e2f[tuple(sorted(e))].append(i)
    adj = defaultdict(set)
    for fs in e2f.values():
        if len(fs) == 2:
            a, b = fs
            adj[a].add(b); adj[b].add(a)
    return adj, e2f

def _face_normals(V: np.ndarray, T: np.ndarray):
    v0, v1, v2 = V[T[:,0]], V[T[:,1]], V[T[:,2]]
    n = np.cross(v1 - v0, v2 - v0)
    n /= (np.linalg.norm(n, axis=1, keepdims=True) + 1e-12)
    return n

def _proj_uv(P: np.ndarray, c: np.ndarray, u: np.ndarray, v: np.ndarray) -> np.ndarray:
    d = P - c
    if d.ndim == 1:
        return np.array([[d @ u, d @ v]], dtype=float)
    return np.stack([d @ u, d @ v], axis=-1)

def _unproj_uv(UV: np.ndarray, c: np.ndarray, u: np.ndarray, v: np.ndarray) -> np.ndarray:
    return c + UV[:, 0:1]*u + UV[:, 1:2]*v

def _fit_plane(P: np.ndarray):
    c = P.mean(0)
    X = P - c
    _, _, Vt = np.linalg.svd(X, full_matrices=False)
    n = Vt[-1]
    u = Vt[0]; u /= (np.linalg.norm(u)+1e-12)
    v = np.cross(n, u); v /= (np.linalg.norm(v)+1e-12)
    return c, n, u, v

# ------------------------ 曲率（cotangent mean curvature） ------------------------

def _cotangent_mean_curvature_vertex(V: np.ndarray, T: np.ndarray) -> np.ndarray:
    """
    返回每个顶点的 |H|（mean curvature magnitude），离散公式：
    Hn_i = 0.5 * Σ_j (cot α_ij + cot β_ij) (v_i - v_j)
    H_i = ||Hn_i|| / (2 A_i) ，A_i 为一环三角形面积的 1/3 累加（barycentric area）
    """
    N = V.shape[0]
    Hn = np.zeros((N, 3), dtype=np.float64)
    A  = np.zeros(N, dtype=np.float64)

    i = T[:,0]; j = T[:,1]; k = T[:,2]
    vi, vj, vk = V[i], V[j], V[k]

    # 每个三角形的两倍面积
    n = np.cross(vj - vi, vk - vi)
    area2 = np.linalg.norm(n, axis=1) + 1e-18
    area = 0.5 * area2

    # 三个角的 cot 值
    def cot(a, b, c):
        # 角 at a, 向量 ab, ac
        ab = b - a; ac = c - a
        return (ab * ac).sum(axis=1) / (np.linalg.norm(np.cross(ab, ac), axis=1) + 1e-18)

    cotA = cot(vi, vj, vk)  # at i
    cotB = cot(vj, vk, vi)  # at j
    cotC = cot(vk, vi, vj)  # at k

    # 边权：边 (i,j) 对应对角角 C 的 cotC，等等；双向累加
    # 对 i 顶点的贡献： (cotB)*(vi - vk) + (cotC)*(vi - vj)
    np.add.at(Hn, i, (cotB[:,None])*(vi - vk) + (cotC[:,None])*(vi - vj))
    np.add.at(Hn, j, (cotC[:,None])*(vj - vi) + (cotA[:,None])*(vj - vk))
    np.add.at(Hn, k, (cotA[:,None])*(vk - vj) + (cotB[:,None])*(vk - vi))

    # 面积分配
    np.add.at(A, i, area/3.0); np.add.at(A, j, area/3.0); np.add.at(A, k, area/3.0)

    H = 0.5*np.linalg.norm(Hn, axis=1) / (A + 1e-18)
    return H  # (N,)

# 面曲率 = 顶点曲率均值
def _face_curvature_from_vertex(Hv: np.ndarray, T: np.ndarray) -> np.ndarray:
    return (Hv[T[:,0]] + Hv[T[:,1]] + Hv[T[:,2]]) / 3.0

# ------------------------ 网格生成 & 裁剪 ------------------------

def _square_grid(bmin: np.ndarray, bmax: np.ndarray, h: float) -> np.ndarray:
    if h <= 1e-18: return np.zeros((0,2))
    xs = np.arange(bmin[0], bmax[0] + 1e-9, h)
    ys = np.arange(bmin[1], bmax[1] + 1e-9, h)
    if xs.size==0 or ys.size==0: return np.zeros((0,2))
    X, Y = np.meshgrid(xs, ys, indexing='xy')
    return np.stack([X.ravel(), Y.ravel()], axis=-1)

def _hex_grid(bmin: np.ndarray, bmax: np.ndarray, h: float) -> np.ndarray:
    if h <= 1e-18: return np.zeros((0,2))
    dy = np.sqrt(3.0)*0.5*h
    ys = np.arange(bmin[1], bmax[1] + 1e-9, dy)
    pts=[]
    for r, y in enumerate(ys):
        x0 = bmin[0] + (0.5*h if (r%2) else 0.0)
        xs = np.arange(x0, bmax[0] + 1e-9, h)
        if xs.size: pts.append(np.stack([xs, np.full_like(xs, y)], -1))
    return np.concatenate(pts, 0) if pts else np.zeros((0,2))

def _mask_pts_in_tris_vec(P: np.ndarray, tris_uv: np.ndarray, block_points: int = 20000, eps: float = 1e-12) -> np.ndarray:
    if P.size == 0 or tris_uv.size == 0:
        return np.zeros(P.shape[0], dtype=bool)
    A = tris_uv[:,0,:]; B = tris_uv[:,1,:]; C = tris_uv[:,2,:]
    v0 = C - A; v1 = B - A
    denom = v0[:,0]*v1[:,1] - v1[:,0]*v0[:,1]
    valid = np.abs(denom) > 1e-20
    A,B,C,v0,v1,denom = A[valid], B[valid], C[valid], v0[valid], v1[valid], denom[valid]
    Tn = A.shape[0]
    keep = np.zeros(P.shape[0], dtype=bool)
    Mb = max(512, min(P.shape[0], int(1_000_000 / max(Tn,1))))
    if block_points is not None: Mb = min(Mb, block_points)
    for s in range(0, P.shape[0], Mb):
        e = min(P.shape[0], s+Mb)
        Pb = P[s:e]
        v2 = Pb[None,:,:] - A[:,None,:]
        cross_v2_v1 = v2[...,0]*v1[:,None,1] - v1[:,None,0]*v2[...,1]
        cross_v0_v2 = v0[:,None,0]*v2[...,1] - v2[...,0]*v0[:,None,1]
        v = cross_v2_v1 / denom[:,None]
        w = cross_v0_v2 / denom[:,None]
        u = 1.0 - v - w
        inside = (u>=-eps)&(v>=-eps)&(w>=-eps)
        keep[s:e] = inside.any(axis=0)
    return keep

# ------------------------ 近似平面片（区域生长） ------------------------

def _region_grow_planar(face_normals: np.ndarray, adj: Dict[int, Iterable[int]],
                        angle_thr_deg: float = 3.0, max_faces: int = 20000):
    """按面法向相差 <= angle_thr 的规则做连通区域生长，得到候选近似平面片列表。"""
    F = face_normals.shape[0]
    visited = np.zeros(F, dtype=bool)
    comps = []
    cos_thr = np.cos(np.deg2rad(angle_thr_deg))
    for s in range(F):
        if visited[s]: continue
        stack=[s]; visited[s]=True; comp=[s]
        n0 = face_normals[s]
        while stack and len(comp) < max_faces:
            u = stack.pop()
            for v in adj.get(u, []):
                if visited[v]: continue
                if np.dot(face_normals[v], face_normals[u]) >= cos_thr:
                    visited[v]=True; stack.append(v); comp.append(v)
        comps.append(np.asarray(comp, dtype=int))
    return comps

# ------------------------ 可变半径 Poisson 磁盘（曲面上近似） ------------------------

def _poisson_disk_mesh_candidates(V: np.ndarray, T: np.ndarray,
                                  h_face: np.ndarray, n_scale: float,
                                  oversample: float = 1.3) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    生成候选点（均匀按面积取样）并附带每点的半径 r = n_scale * h_face[fi]
    返回: pts(M,3), r(M,), face_ids(M,)
    """
    v0, v1, v2 = V[T[:,0]], V[T[:,1]], V[T[:,2]]
    area = 0.5*np.linalg.norm(np.cross(v1 - v0, v2 - v0), axis=1)
    # 每面期望采样数 ~ area / cell_area；六边形单元面积系数：
    cell_factor = np.sqrt(3)/2
    n_face = oversample * (area / (cell_factor*np.maximum(h_face**2, 1e-18)))
    n_face = np.ceil(n_face).astype(int)

    # 生成
    idxs = np.repeat(np.arange(T.shape[0]), n_face)
    if idxs.size == 0:
        return np.zeros((0,3)), np.zeros((0,)), np.zeros((0,), int)
    r = n_scale * h_face[idxs]
    # 随机重心采样
    r1 = np.random.rand(idxs.size)
    r2 = np.random.rand(idxs.size)
    flip = (r1 + r2) > 1.0
    r1[flip] = 1.0 - r1[flip]
    r2[flip] = 1.0 - r2[flip]
    a = 1.0 - r1 - r2; b = r1; c = r2
    A, B, C = V[T[idxs,0]], V[T[idxs,1]], V[T[idxs,2]]
    pts = a[:,None]*A + b[:,None]*B + c[:,None]*C
    return pts, r, idxs

def _poisson_thinning_variable_radius(pts: np.ndarray, radii: np.ndarray,
                                      bbmin: np.ndarray, cell: float) -> np.ndarray:
    """
    简单的 3D 空间哈希；变量半径规则：两点最小间距 >= 0.5*(r_i + r_j)
    返回保留的索引下标
    """
    if pts.shape[0] == 0:
        return np.zeros((0,), dtype=int)
    grid = defaultdict(list)
    inv = 1.0 / max(cell, 1e-18)
    order = np.random.permutation(pts.shape[0])
    keep = []
    for idx in order:
        p = pts[idx]
        r = radii[idx]
        gx, gy, gz = np.floor((p - bbmin) * inv).astype(int)
        ok = True
        for dx in (-1,0,1):
            for dy in (-1,0,1):
                for dz in (-1,0,1):
                    key = (gx+dx, gy+dy, gz+dz)
                    if key not in grid: continue
                    nb_idx = grid[key]
                    if len(nb_idx)==0: continue
                    q = pts[nb_idx]
                    rq = radii[nb_idx]
                    d2 = np.sum((q - p)**2, axis=1)
                    min_allowed = 0.25*(r + rq)**2  # (0.5*(r_i+r_j))^2
                    if np.any(d2 < min_allowed):
                        ok = False; break
                if not ok: break
            if not ok: break
        if ok:
            keep.append(idx)
            grid[(gx,gy,gz)].append(idx)
    return np.array(keep, dtype=int)

# ------------------------ 主函数 ------------------------

def mesh_to_point_cloud_adaptive_grid(
    mesh: o3d.geometry.TriangleMesh,
    *,
    grid_type: str = "hex",             # "hex" / "square"：平面片晶格类型
    target_points: Optional[int] = 200_000,  # 全局目标点数（强烈建议给）
    flat_normal_thr_deg: float = 3.0,        # 平面片区域生长的法向阈值
    planarity_tol_ratio: float = 1e-3,       # 平面拟合容差（相对对角线）
    curvature_power: float = 2.0,            # 曲率→步长非线性（调大=更强加密）
    h_min_scale: float = 0.7,                # 自动步长：h_min = h_avg * 0.7
    h_max_scale: float = 1.4,                # 自动步长：h_max = h_avg * 1.4
    boundary_boost: float = 1.4,             # 边界面片曲率放大倍率
    debug: bool = False
) -> o3d.geometry.PointCloud:

    assert isinstance(mesh, o3d.geometry.TriangleMesh)
    V = np.asarray(mesh.vertices, float)
    T = np.asarray(mesh.triangles, int)
    F = T.shape[0]
    if F == 0:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(V.copy())
        return pcd

    # --- 尺度统计 ---
    v0, v1, v2 = V[T[:,0]], V[T[:,1]], V[T[:,2]]
    face_area = 0.5*np.linalg.norm(np.cross(v1 - v0, v2 - v0), axis=1)
    S = float(face_area.sum())
    bbmin, bbmax = V.min(0), V.max(0)
    diag = float(np.linalg.norm(bbmax - bbmin))

    # 估算平均步长（按六边形单元面积）
    cell_factor = np.sqrt(3)/2
    if target_points is None or target_points <= 0:
        target_points = int(max(1.0, S / (cell_factor*(diag*0.02)**2)))
    h_avg = np.sqrt(max(1e-12, S / (cell_factor*target_points)))
    h_min = h_avg * h_min_scale
    h_max = h_avg * h_max_scale
    if debug:
        print(f"[auto] area={S:.2f}, diag={diag:.2f}, h_avg={h_avg:.4f}, h_min={h_min:.4f}, h_max={h_max:.4f}")

    # --- 曲率（cotangent mean curvature），更能反映“平滑曲面弯曲度” ---
    Hv = _cotangent_mean_curvature_vertex(V, T)              # (N,)
    Hf = _face_curvature_from_vertex(Hv, T)                  # (F,)
    adj, e2f = _face_adjacency(T)
    fn = _face_normals(V, T)

    # 边界加权（促使边缘更密）
    if boundary_boost and boundary_boost > 1.0:
        boundary_faces = [fs[0] for fs in e2f.values() if len(fs)==1]
        if len(boundary_faces):
            Hf[np.asarray(boundary_faces, int)] *= float(boundary_boost)

    # 曲率 → 面步长
    c_lo = np.percentile(Hf, 20)   # 放宽区间，拉开差异
    c_hi = np.percentile(Hf, 95)
    x = np.clip((Hf - c_lo) / max(1e-12, (c_hi - c_lo)), 0.0, 1.0)
    w = x ** curvature_power
    h_face = h_max - (h_max - h_min) * w  # 高曲率→更接近 h_min

    # --- 近似平面片（区域生长 + 平面拟合） ---
    planar_comps = _region_grow_planar(fn, adj, angle_thr_deg=flat_normal_thr_deg)
    grid_fn = _hex_grid if grid_type == "hex" else _square_grid
    plane_tol = max(1e-9, planarity_tol_ratio * diag)

    pts_all = []
    used_face = np.zeros(F, dtype=bool)

    for comp in planar_comps:
        vids = np.unique(T[comp].ravel())
        P = V[vids]
        c, n, u, v = _fit_plane(P)
        # 拟合残差（最大离平面距离）
        if np.max(np.abs((P - c) @ n)) > plane_tol:
            continue  # 不是近似平面，留给曲面采样
        # 片内 UV 三角形
        tris_uv = np.stack([_proj_uv(V[T[fi]], c, u, v) for fi in comp], 0)  # (Tp,3,2)
        uv_all = tris_uv.reshape(-1,2)
        bmin, bmax = uv_all.min(0) - 1e-9, uv_all.max(0) + 1e-9
        # 片均值步长
        h_patch = float((h_face[comp]*face_area[comp]).sum() / (face_area[comp].sum()+1e-18))
        Pgrid = grid_fn(bmin, bmax, h_patch)
        if Pgrid.size == 0: 
            continue
        keep = _mask_pts_in_tris_vec(Pgrid, tris_uv, block_points=25000)
        UV = Pgrid[keep]
        if UV.size:
            pts_all.append(_unproj_uv(UV, c, u, v))
            used_face[comp] = True

    # --- 非平面（曲面）区域：可变半径 Poisson 磁盘采样 ---
    rest = np.where(~used_face)[0]
    if rest.size:
        pts_cand, r_cand, fi_cand = _poisson_disk_mesh_candidates(V, T[rest], h_face[rest], n_scale=1.0, oversample=1.3)
        if pts_cand.size:
            # 空间哈希网格尺寸：按最小半径设置（更保守）
            rmin = float(np.maximum(r_cand.min(), 1e-9))
            cell = rmin/np.sqrt(3.0)
            keep_idx = _poisson_thinning_variable_radius(pts_cand, r_cand, bbmin, cell)
            if keep_idx.size:
                pts_all.append(pts_cand[keep_idx])

    if not pts_all:
        return o3d.geometry.PointCloud()
    P = np.concatenate(pts_all, axis=0)

    # 轻量去重（量化）
    q = max(1e-9, h_min*1e-3)
    key = np.round((P - bbmin)/q).astype(np.int64)
    _, idx = np.unique(key, axis=0, return_index=True)
    P = P[np.sort(idx)]

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(P.astype(np.float64, copy=False))
    if debug:
        print(f"[result] points={len(P)}, planar_patches={sum(used_face)>0}")
    return pcd


# ----------------------------- 用法示例 -----------------------------
# 建议：用 target_points 控制整体点数，绘图前做可视化降采样（Plotly 对超大点集非常慢）
pcd_2 = mesh_to_point_cloud_adaptive_grid(
    mesh2,
    grid_type="hex",
    target_points=200_000,      # 先控量，之后再加密
    flat_normal_thr_deg=3.0,    # 放宽/收紧可试 2~5
    planarity_tol_ratio=2e-3,   # 稍放宽更容易形成平面片
    curvature_power=2.0,
    h_min_scale=0.6,
    h_max_scale=1.5,
    boundary_boost=1.5,
    debug=True
)
# # 可视化：先降采样到 2~3e5 再 plotly，避免界面卡死
P = np.asarray(pcd_2.points)
if P.shape[0] > 30_000:
    keep = 30_000 / P.shape[0]
    pcd_vis = pcd_2.random_down_sample(keep)
else:
    pcd_vis = pcd_2
o3d.visualization.draw_plotly([mesh2, pcd_vis], width=1000, height=600)


In [4]:
import os
import glob
import open3d as o3d
from tqdm.notebook import tqdm
import logging
import torch
from scipy.spatial import KDTree

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 数据集根目录
root = "/data/Real3D-AD-PCD"

def gen_pcd(root, category, file_path):

    class_names = ['good', 'bulge', 'sink']
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    # 构造输出路径
    output_dir = os.path.join(root, category, 'test_neo')
    output_path = os.path.join(output_dir, os.path.basename(file_path))

    # 检查输出文件是否已存在
    # if os.path.exists(output_path):
    #     logging.info(f"文件已存在，跳过: {output_path}")
    #     return

    # 错误呋喃类
    fname = os.path.basename(file_path).lower()
    defect_type = 'good'
    for d in class_names[1:]:
        if d in fname:
            defect_type = d
            break

    label = class_to_idx[defect_type]

    # 加载点云
    pcd = o3d.io.read_point_cloud(file_path)

    # 调用 point_cloud_to_mesh_2
    mesh2 = point_cloud_to_mesh_2(pcd)

    # 调用 mesh_to_point_cloud
    pcd_2 = mesh_to_point_cloud(mesh2, n_global=8192, n_feature=8192)

    # ======= 生成带标签的 PCD 文件 =======
    points = np.asarray(pcd_2.points)
    span = points.max(axis=0) - points.min(axis=0)
    # print(f"x span: {span[0]:.6f}, y span: {span[1]:.6f}, z span: {span[2]:.6f}")

    # 读取Ground Truth 生成掩码
    gt_path = file_path.replace('/test/', '/gt/').replace('.pcd', '.txt')

    mask = parse_gt(pcd_2, label, gt_path)  # 这里应传 pcd_2, label, gt_path
    mask_np = mask.cpu().numpy().reshape(-1)

    assert points.shape[0] == mask_np.shape[0], "点数和掩码长度不一致"

    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)

    # 保存处理后的点云
    with open(output_path, 'w') as f:
        f.write(
            "# .PCD v.7 - Point Cloud Data file format\n"
            "VERSION .7\n"
            "FIELDS x y z label\n"
            "SIZE 4 4 4 4\n"
            "TYPE F F F I\n"
            "COUNT 1 1 1 1\n"
            f"WIDTH {len(points)}\n"
            "HEIGHT 1\n"
            "VIEWPOINT 0 0 0 1 0 0 0\n"
            f"POINTS {len(points)}\n"
            "DATA ascii\n"
        )
        for (x, y, z), l in zip(points, mask_np):
            f.write(f"{x} {y} {z} {int(l)}\n")


def parse_gt(pcd, label, gt_path, distance_threshold=0.2):

    pcd_points = np.asarray(pcd.points)
    anomaly_mask = torch.zeros(len(pcd_points), dtype=torch.long)

    #print(label)

    # 正常的样本就不用分析了
    if 0 == label:
        return anomaly_mask

    with open(gt_path, 'r') as f:
        lines = f.readlines()

    # 使用 KDTree 进行批量查询
    tree = KDTree(pcd_points)
    anomalies = []

    # 批量处理坐标点
    coords_list = []
    for line in lines:
        vals = line.strip().split()
        if len(vals) >= 4 and float(vals[-1]) == 1.0:
            coords_list.append([float(vals[0]), float(vals[1]), float(vals[2])])

    # 如果有需要查询的坐标
    if coords_list:
        coords_array = np.array(coords_list)
        distances, indices = tree.query(coords_array)  # 批量查询
        for i, dist in enumerate(distances):
            # 如果最近邻距离小于阈值，则认为该点有效
            valid_indices = indices[i][dist < distance_threshold]
            anomaly_mask[valid_indices] = label  # 设置对应的 mask

    return anomaly_mask

#gen_pcd(root, 'starfish', '/data/Real3D-AD-PCD/starfish/test/506_bulges_cut.pcd')
# gen_pcd(root, 'airplane', '/data/Real3D-AD-PCD/airplane/test/536_bulge.pcd')
# gen_pcd(root, 'toffees', '/data/Real3D-AD-PCD/toffees/test/564_sink.pcd')

In [ ]:
# 只显示错误，不显示 warning/info
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

all_files = []
for category in sorted(os.listdir(root)):
    category_path = os.path.join(root, category)
    if not os.path.isdir(category_path):
        continue
    category_files = sorted(glob.glob(os.path.join(category_path, 'test', "*.pcd")))
    # 过滤掉文件名中带 hybrid 的文件
    filtered_files = [file for file in category_files if "hybrid" not in os.path.basename(file)]
    all_files.extend([(category, file) for file in category_files])


def task_wrapper(file_info):
    category, file_path = file_info
    try:
        gen_pcd(root, category, file_path)
        return (file_path, "成功")
    except Exception as e:
        return (file_path, f"失败: {e}")
        
# 开始串行处理
logging.info(f"数据集大小: {len(all_files)}，使用单线程处理")


for file_info in tqdm(all_files, desc="单线程处理"):
    file_path, status = task_wrapper(file_info)
    logging.info(f"{file_path} 处理状态: {status}")